# Unit 3 exercises: predict where the car will be

A controller that can predict the next second can brake before the wall instead of after it.
These exercises build that prediction from nothing: the bicycle's circle, the longitudinal map, Euler and RK4, their measured error orders, and then real telemetry.
Every number you produce here is a number about racekit's **kinematic car**: a model, not the truth (Unit 8 re-identifies the parameters from data).

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Functions you write in one exercise are reused in later ones, so run the exercises in order.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from racekit.cars import KinematicCar
from racekit.runner import CONTROL_DT, run_race
from racekit.tracks import load_track

STARTER = Path("../../projects/racekit/agents/unit-01.py")

# racekit's kinematic car (a model, not the truth)
WHEELBASE = 2.875  # m
DELTA_MAX = math.radians(70.0)  # wheel angle at |steer| = 1; delta = -steer * DELTA_MAX
A_THROTTLE = 6.0  # m/s^2 at full throttle
A_BRAKE = 8.0  # m/s^2 at full brake, against the motion
C_DRAG = 4.0e-4  # 1/m: drag deceleration C_DRAG * v * |v|

## Exercise 1 - The circle, and the steer a corner needs

A positive steer turns right; the wheel angle is δ = −steer · δ_max, and the path's signed curvature is κ = tan δ / L (left positive).

1. For **steer = −0.1** at **v = 15 m/s**, work out the steady-state circle.
2. For each of `"oval"`, `"test-circuit"` and `"monza-like"`, compute the signed **Menger curvature** (Unit 2) at every waypoint i from the triple (i − 1, i, i + 1), wrapping cyclically at both ends (index arithmetic with `% n` does the wrap).
   Take the waypoint with the largest |κ| and find the steer that makes the kinematic car's path curvature equal to it.
   (Use `math.atan2(y, 1.0)` for arctan y, as the lesson does: `atan` is not on the book's list of callables.)

Define:

- `delta`: the wheel angle for steer −0.1 (rad, signed);
- `radius`: the turn radius (m, positive);
- `yaw_rate`: ψ̇ at 15 m/s (rad/s, signed: counter-clockwise positive);
- `turns_left`: `True` when this steer turns the car left;
- `lap_time`: the time to drive once round that circle at 15 m/s (s);
- `steer_needed`: a dict from track name to the signed steer for that track's tightest corner (a right turn needs a positive steer).

In [ ]:
delta = None  # TODO
radius = None  # TODO
yaw_rate = None  # TODO
turns_left = None  # TODO
lap_time = None  # TODO
steer_needed = {}  # TODO

## Exercise 2 - The longitudinal map and the terminal speed

Write `acceleration(v, throttle, brake)`: a_T·throttle, minus a_B·brake **against the motion** (subtract it when v > 0, add it when v < 0, nothing at v = 0), minus the drag C_DRAG·v·|v|.

1. Find the terminal speed at full throttle, analytically.
2. Find the speed where the starter's speed controller settles on a straight, by solving the quadratic.
   The controller is `throttle = clip(0.05 * (20 - v), 0, 1)` and `brake = clip(-0.05 * (20 - v), 0, 1)`.
3. Check it by simulation: integrate **only the speed** from v = 0 with that controller re-evaluated every step, using RK4 on v̇ = `acceleration(v, throttle, brake)` with h = 0.05 s and the action held within each step, for 1200 steps (60 s).
4. Brake from 20 m/s with brake = 1 and throttle = 0 for 2.0 s (40 RK4 steps of 0.05 s, integrating x and v along a straight line) and record the distance.
   Compare it with the no-drag distance from constant deceleration a_B.

Define:

- `acceleration`: the function;
- `v_terminal`: the full-throttle terminal speed (m/s);
- `v_equilibrium`: the root of the quadratic (m/s);
- `v_after_60s`: the simulated speed after 1200 steps (m/s);
- `brake_distance`: the simulated distance in 2.0 s of full braking from 20 m/s (m);
- `brake_distance_no_drag`: the same with no drag, from the constant-deceleration formula (m);
- `why_below_target`: one sentence on why the starter settles below 20 m/s, and which Unit 1 number this is.

In [ ]:
def acceleration(v, throttle, brake):
    return None  # TODO


v_terminal = None  # TODO
v_equilibrium = None  # TODO
v_after_60s = None  # TODO
brake_distance = None  # TODO
brake_distance_no_drag = None  # TODO
why_below_target = "TODO"

## Exercise 3 - Euler's half-step lag

Write `derivatives(state, action)` for the full model, with `state = np.array([x, y, psi, v])` and `action = (throttle, steer, brake)`, returning `np.array([x', y', psi', v'])`; then write `euler_step(state, action, h)`.

Drive a steady circle at **steer = 0.1** (R = L / tan(0.1·δ_max), a right turn) with the drag-balancing throttle C_DRAG·v²/A_THROTTLE, from the state (0, 0, 0, v).
The true centre is (0, −R).
Run for 100 s, that is `int(round(100 / h))` Euler steps, and take the **maximum over all steps** of |distance from the true centre − R|.
Do this for three cases: (h, v) = (0.05, 10), (0.025, 10) and (0.05, 20).

**Predict first** from the lesson's half-step-lag argument, then measure.

Define:

- `derivatives`, `euler_step`: the functions;
- `predicted_shifts`: a list of your three predictions (m), in the order of the cases, written as expressions;
- `measured_shifts`: a list of the three measured maxima (m);
- `scaling`: one sentence on how the shift scales with h and with v, and why it does not grow with the number of laps.

In [ ]:
def derivatives(state, action):
    return None  # TODO


def euler_step(state, action, h):
    return None  # TODO


CASES = [(0.05, 10.0), (0.025, 10.0), (0.05, 20.0)]
predicted_shifts = []  # TODO: before measuring
measured_shifts = []  # TODO
scaling = "TODO"

## Exercise 4 - Measure the orders on a new manoeuvre

Write `rk4_step(state, action, h)`, the classical fourth-order Runge–Kutta step with the action held.

The case: **throttle 1.0, steer −0.2, from rest** at state (0, 0, 0, 0), for **T = 3 s**.
For a step h, take `int(round(T / h))` steps.
The reference is RK4 at h = 10⁻³ s.
The error is the distance between the final (x, y) and the reference's final (x, y).

Define:

- `rk4_step`: the function;
- `euler_errors`, `rk4_errors`: NumPy arrays of the errors (m) for h = 0.2, 0.1, 0.05 and 0.025, in that order;
- `euler_orders`, `rk4_orders`: NumPy arrays of the three observed orders log₂(e(h) / e(h/2)), in the same order;
- `fig`, `ax`: one log–log plot of both error curves against h, with axis labels, a title and a legend;
- `ticks_for_euler`: how many times more steps than at h = 0.025 Euler would need to match RK4's error at h = 0.025, assuming its order is exactly 1 (a float).

In [ ]:
def rk4_step(state, action, h):
    return None  # TODO


STEPS = np.array([0.2, 0.1, 0.05, 0.025])
euler_errors = None  # TODO
rk4_errors = None  # TODO
euler_orders = None  # TODO
rk4_orders = None  # TODO
fig, ax = None, None  # TODO
ticks_for_euler = None  # TODO

## Exercise 5 - Replay the telemetry, with and without noise

Race the starter on `"test-circuit"` twice: `clean` with `KinematicCar(jitter=0.0)`, and `noisy` with `KinematicCar(jitter=0.05)` and `seed=0`.

For each race, take the start rows `np.arange(100, len(telemetry) - 20, 50)`.
From each start row's recorded (x, y, yaw, v), apply the recorded actions of the next 20 rows (rows start + 1 … start + 20, each held for one tick of 0.05 s) and compare with the recorded state at row start + 20.
The position error is the distance between the (x, y) positions.

Define:

- `clean`, `noisy`: the two `RaceResult`s;
- `rk4_max_clean`, `euler_max_clean`: the largest position error on `clean` with one RK4 step per tick, and with one Euler step per tick (m);
- `euler10_max_clean`: the same with Euler in 10 substeps of 0.005 s per tick (m);
- `max_yaw_error_clean`: the largest heading error of the RK4 replay on `clean`, **wrapped** to [−π, π) before taking the absolute value (rad);
- `rk4_max_noisy`, `rk4_median_noisy`: the largest and the median RK4 position error on `noisy` (m);
- `why_noisy`: one sentence explaining why a perfect integrator of the right model is metres off on the jittered race. (Reread the Unit 1 lesson's description of the telemetry's action columns.)

In [ ]:
clean = None  # TODO
noisy = None  # TODO
rk4_max_clean = None  # TODO
euler_max_clean = None  # TODO
euler10_max_clean = None  # TODO
max_yaw_error_clean = None  # TODO
rk4_max_noisy = None  # TODO
rk4_median_noisy = None  # TODO
why_noisy = "TODO"

## Exercise 6 - Where will the car be relative to the track?

A real prediction does not know the future actions.
On the `clean` race of Exercise 5, with the same start rows, predict 1 s ahead by holding the **first** action (the action of row start + 1) for all 20 ticks, with one RK4 step per tick.
Then describe each prediction's miss in the track's terms, with Unit 2's tools:

- **frames:** rotate the error vector (predicted − recorded position at row start + 20) into the **car frame** of the recorded state at row start + 20, giving a forward (along-heading) and a left (lateral) component;
- **cross-track error:** write `cross_track_error(px, py, xs, ys, hint, reach=8)` exactly as Unit 2 pinned it (segments hint − 8 … hint + 8, cyclic; the foot parameter clamped to [0, 1] to choose the segment, nearest wins, ties to the first; offset = (dx·e_y − dy·e_x) / |d| with e = p − a), and apply it to the **predicted** position with `hint` = the index of the waypoint nearest the **recorded** position at row start + 20 (search all waypoints).

Define:

- `cross_track_error`: the function;
- `along_rms`, `lateral_rms`: the root-mean-square of the forward and the left error components over all windows (m);
- `max_predicted_offset`: the largest |predicted cross-track error| (m);
- `max_offset_gap`: the largest |predicted cross-track error − the recorded `offset` at row start + 20| (m);
- `dominant`: `"along"` or `"lateral"`, whichever RMS is larger;
- `what_it_means`: one sentence on what a controller that trusted this held-action prediction would conclude near a corner, given that the narrowest half-width of `test-circuit` is `min(load_track("test-circuit").half_width)`.

In [ ]:
def cross_track_error(px, py, xs, ys, hint, reach=8):
    return None  # TODO: return (segment index, signed offset)


along_rms = None  # TODO
lateral_rms = None  # TODO
max_predicted_offset = None  # TODO
max_offset_gap = None  # TODO
dominant = "TODO"
what_it_means = "TODO"

## Exercise 7 - Challenge: how small must Euler's step be?

The case: from (0, 0, 0, 20) hold throttle 0, steer 0.1 and brake 0 for **1 s**.
The reference is RK4 with 10 000 steps of 10⁻⁴ s.
Use Euler with h = 1/n s for a whole number n of steps.

1. Measure Euler's 1 s position error at the race tick (n = 20).
2. **Predict** the smallest n that brings the error below 1 cm, assuming the error is exactly proportional to h.
   (Round up with integer arithmetic, for example `int(x) + 1` when x is not a whole number: `ceil` is not on the book's list of callables.)
3. Find the true smallest n by measurement (the error falls monotonically in n here, so a bisection is enough).
4. Measure RK4's error at n = 20 for comparison.

Define:

- `error_at_tick`: Euler's error at n = 20 (m);
- `n_predicted`: the smallest integer n the first-order law predicts;
- `n_actual`: the measured smallest n with error < 0.01 m;
- `h_star`: the step size 1 / `n_actual` (s);
- `rk4_error_at_tick`: RK4's error at n = 20 (m);
- `cost_verdict`: one sentence comparing Euler's cost for 1 cm with RK4's cost at the race tick (RK4 makes 4 evaluations of f per step, Euler 1), and saying whether the prediction was off and why it could be.

In [ ]:
error_at_tick = None  # TODO
n_predicted = None  # TODO
n_actual = None  # TODO
h_star = None  # TODO
rk4_error_at_tick = None  # TODO
cost_verdict = "TODO"

## Exercise 8 - Challenge: semi-implicit Euler, and its order

**Semi-implicit Euler** updates the speed and the heading first, then moves the car with the **new** values:

$$v_{n+1} = v_n + h\,\dot v(\mathbf s_n), \quad \psi_{n+1} = \psi_n + h\,\dot\psi(\mathbf s_n), \quad x_{n+1} = x_n + h\,v_{n+1}\cos\psi_{n+1}, \quad y_{n+1} = y_n + h\,v_{n+1}\sin\psi_{n+1}.$$

It costs the same as Euler: one evaluation of f per step.

1. Write `semi_implicit_step(state, action, h)`.
2. Measure its order on the lesson's convergence case: throttle 0.6, steer 0.15, brake 0, from (0, 0, 0, 5) for T = 4 s, `int(round(T / h))` steps, reference RK4 at h = 10⁻⁴, for h = 0.4, 0.2, 0.1 and 0.05.
3. Run it on Exercise 3's steady circle at steer 0.1, v = 10, h = 0.05 for **2000** steps, and take the maximum |distance from the true centre − R|.

Define:

- `semi_implicit_step`: the function;
- `semi_errors`, `semi_orders`: NumPy arrays of the four errors (m) and the three observed orders;
- `euler_errors_lesson`: plain Euler's four errors on the same case (m);
- `constant_ratio`: plain Euler's error over semi-implicit Euler's at h = 0.05;
- `semi_circle_shift`: the circle's maximum |distance − R| (m);
- `order_verdict`: one or two sentences: is semi-implicit Euler a higher-order method, is it better here, and what does the circle show?

In [ ]:
def semi_implicit_step(state, action, h):
    return None  # TODO


semi_errors = None  # TODO
semi_orders = None  # TODO
euler_errors_lesson = None  # TODO
constant_ratio = None  # TODO
semi_circle_shift = None  # TODO
order_verdict = "TODO"